## Schizophrenia Symptoms

In [1]:
# import important libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

### 1. Load the dataset

In [2]:
df = pd.read_csv("../data/SchizophreniaSymptomnsData.csv")

df.head()

,Name,Age,Gender,Marital_Status,Fatigue,Slowing,Pain,Hygiene,Movement,Schizophrenia
0,Leslie Goodwin,68,Female,Single,0.698075,0.123064,0.375303,0.234639,0.251869,Elevated Proneness
1,Dr. Troy Castaneda,88,Male,Married,0.049245,-0.042080,0.432807,0.501238,0.379948,Moderate Proneness
2,Chelsey Allen,67,Female,Married,0.651995,0.187117,NaN,0.301942,0.302588,Elevated Proneness
3,Dr. Devin Skinner DVM,95,Female,Widowed,0.036324,0.580808,0.005356,0.306968,0.813618,Moderate Proneness
4,Megan Mendez,81,Female,Widowed,0.926727,0.484202,0.702405,0.736054,0.579448,High Proneness


### 2. Inspect the data

1. `Name` is irrelvant for the prediction, it needs to be removed.
2. `Age` might be an important factor in the prediction of the model. `StandardScaler` will be needed as the values of the age are quit big compared to `fatigue`, `slowing`, `pain`, `hygiene`, `movement`.
3. `Gender` & `Material_Status` are both `str`, therefore OneHotEncoder will need to be used to turn into either binary or multi-class encoders.\
4. `Schizophrenia` is the predicted attributed. 

About the missing values there are two ways of dealing with them:
1. Use an Imputer to refill the values of the missing attr values.
2. Or remove these samples completely.
We have an average of 237 values in this dataset, I will resume with the first options as i prefer not removing this much data.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 10 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Name            5000 non-null   str    
 1   Age             5000 non-null   int64  
 2   Gender          5000 non-null   str    
 3   Marital_Status  5000 non-null   str    
 4   Fatigue         4756 non-null   float64
 5   Slowing         4771 non-null   float64
 6   Pain            4758 non-null   float64
 7   Hygiene         5000 non-null   float64
 8   Movement        5000 non-null   float64
 9   Schizophrenia   5000 non-null   str    
dtypes: float64(5), int64(1), str(4)
memory usage: 390.8 KB


## 3. Explore the data EDA

This will include include forming an interest in the dataset.

1. multi-class matirial-status single, married, widowed.
2. Check missing values (already done).
3. With values we can check `age` validity.
4. Checking highest age. **Result:** the highest age is 95, which is normal, plotting will determine if it is a outlier or not.
5. Check `Name` attr for any duplicates.
6. Check the number female and male in this dataset

In [4]:
df.head()

,Name,Age,Gender,Marital_Status,Fatigue,Slowing,Pain,Hygiene,Movement,Schizophrenia
0,Leslie Goodwin,68,Female,Single,0.698075,0.123064,0.375303,0.234639,0.251869,Elevated Proneness
1,Dr. Troy Castaneda,88,Male,Married,0.049245,-0.042080,0.432807,0.501238,0.379948,Moderate Proneness
2,Chelsey Allen,67,Female,Married,0.651995,0.187117,NaN,0.301942,0.302588,Elevated Proneness
3,Dr. Devin Skinner DVM,95,Female,Widowed,0.036324,0.580808,0.005356,0.306968,0.813618,Moderate Proneness
4,Megan Mendez,81,Female,Widowed,0.926727,0.484202,0.702405,0.736054,0.579448,High Proneness


In [5]:
df.tail()

,Name,Age,Gender,Marital_Status,Fatigue,Slowing,Pain,Hygiene,Movement,Schizophrenia
4995,Anna Blevins,61,Female,Single,0.933016,0.505532,1.010435,0.868590,0.552105,High Proneness
4996,Robert Frazier,60,Female,Single,0.260125,0.625811,-0.076161,0.079046,0.593206,Moderate Proneness
4997,Louis Flores,62,Male,Married,0.248583,NaN,0.596990,0.119659,0.782998,Moderate Proneness
4998,Julie Nguyen,76,Female,Widowed,0.265702,0.525682,0.546284,0.402468,0.222236,Moderate Proneness
4999,Krista Cunningham,59,Male,Married,0.489590,0.630955,0.128390,0.552271,0.580275,Elevated Proneness


In [6]:
df.sample()

,Name,Age,Gender,Marital_Status,Fatigue,Slowing,Pain,Hygiene,Movement,Schizophrenia
2291,Gerald Salas,84,Male,Widowed,0.394094,0.569767,0.607454,0.118943,0.718108,Elevated Proneness


In [7]:
df.describe()

,Age,Fatigue,Slowing,Pain,Hygiene,Movement
count,5000.00000,4756.000000,4771.000000,4758.000000,5000.000000,5000.000000
mean,74.83340,0.503835,0.499524,0.499612,0.499717,0.499952
std,9.57787,0.296123,0.295365,0.294886,0.294907,0.289860
min,55.00000,-0.095115,-0.094843,-0.095771,-0.094284,-0.089272
25%,67.00000,0.247056,0.245795,0.251174,0.248998,0.254143
50%,75.00000,0.506278,0.502403,0.498051,0.501223,0.503340
75%,83.00000,0.759499,0.744812,0.749946,0.751029,0.741253
max,95.00000,1.091136,1.092146,1.090027,1.086922,1.088914


In [9]:
df[df["Age"] <= 0].values

array([], shape=(0, 10), dtype=object)

In [10]:
df["Age"].max()

np.int64(95)

In [20]:
df["Name"].duplicated().count()
# duplicated gives false values so there are no duplication
print(f"unique values: {df["Name"].duplicated().sum()}")
# df[df["Name"].duplicated(keep=False)]}

unique values: 159


In [25]:
print(f"The number of Females is {(df['Gender'] == "Female").sum()}")
print(f"The number of Males is {(df['Gender'] == "Male").sum()}")

The number of Females is 2510
The number of Males is 2490
